<a href="https://colab.research.google.com/github/mswargwani/Blueprint-AI-Estimator/blob/main/Takeoff_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q roboflow ultralytics

import os
import glob
import yaml
from roboflow import Roboflow
from ultralytics import YOLO

rf = Roboflow(api_key="IwoXx5jxy88krroidIkA")
project = rf.workspace("floor-plan-rendering").project("floor-plan-ai-object-detection")
version = project.version(1)
dataset = version.download("yolov8")

yaml_path = os.path.join(dataset.location, "data.yaml")

with open(yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

old_names = data_cfg["names"]
merge_map = {
    "Sofa_1P": "Sofa 1P",
    "Sofa_2P": "Sofa 2P",
    "Sofa_3P": "Sofa 3P",
}

new_names = []
for name in old_names:
    clean_name = merge_map.get(name, name)
    if clean_name not in new_names:
        new_names.append(clean_name)

id_remap = {
    old_idx: new_names.index(merge_map.get(name, name))
    for old_idx, name in enumerate(old_names)
}

for split in ["train", "valid", "test"]:
    label_dir = os.path.join(dataset.location, split, "labels")
    if not os.path.exists(label_dir):
        continue
    for txt_file in glob.glob(os.path.join(label_dir, "*.txt")):
        updated_lines = []
        with open(txt_file, "r") as lf:
            for line in lf:
                parts = line.strip().split()
                if parts:
                    parts[0] = str(id_remap[int(parts[0])])
                    updated_lines.append(" ".join(parts))
        with open(txt_file, "w") as lf:
            lf.write("\n".join(updated_lines) + "\n")

data_cfg["names"] = new_names
data_cfg["nc"] = len(new_names)
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f)

print(f"Ontology Normalized: {len(old_names)} raw classes merged into {len(new_names)} clean classes.")

model = YOLO("yolo11s.pt")

results = model.train(
    data=yaml_path,
    epochs=100,
    patience=20,
    imgsz=800,
    batch=32,
    device=[0, 1],
    freeze=10,
    optimizer="AdamW",
    lr0=0.001,
    mixup=0.15,
    copy_paste=0.30,
    close_mosaic=15,
    project="Engine",
    name="yolo11s_enterprise_25cls",
    exist_ok=True,
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Floor-Plan-AI-Object-Detection-1 in yolov8:: 100%|██████████| 20680/20680 [00:01<00:00, 13871.84it/s]


Ontology Normalized: 28 raw classes merged into 25 clean classes.
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Floor-Plan-AI-Object-Detection-1/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=None, kobj=1.0, line_width=None, lr0=0.001, lrf=0.

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      4.86G     0.8508      2.108      1.644         21        800: 100% ━━━━━━━━━━━━ 267/267 3.2it/s 1:240.2sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.5it/s 7.5s0.3s
                   all       1208       1266      0.412      0.438      0.314      0.138

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      4.87G       0.76      1.893      1.545         33        800: 100% ━━━━━━━━━━━━ 267/267 3.2it/s 1:230.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.6it/s 7.3s0.4s
                   all       1208       1266      0.415      0.484      0.399      0.234

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      4.87G     0.7266      1.779      1.511         24        8

In [ ]:
import os
import pandas as pd
from ultralytics import YOLO

weights_path = "/kaggle/working/runs/detect/Engine/yolo11s_enterprise_25cls/weights/best.pt"
model = YOLO(weights_path)

metrics = model.val(data=yaml_path, imgsz=800, split="val", verbose=False)

rows = []
for i, cls_idx in enumerate(metrics.box.ap_class_index):
    cls_name = model.names[int(cls_idx)]
    p, r, ap50, ap50_95 = metrics.box.class_result(i)
    f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0

    rows.append({
        "Class ID": int(cls_idx),
        "Class Name": cls_name,
        "Precision": round(float(p), 4),
        "Recall": round(float(r), 4),
        "F1-Score": round(float(f1), 4),
        "mAP50": round(float(ap50), 4),
        "mAP50-95": round(float(ap50_95), 4),
    })

df = pd.DataFrame(rows).sort_values(by="mAP50-95", ascending=False).reset_index(drop=True)

overall = pd.DataFrame([{
    "Class ID": "-",
    "Class Name": "ALL CLASSES (Overall)",
    "Precision": round(float(metrics.box.mp), 4),
    "Recall": round(float(metrics.box.mr), 4),
    "F1-Score": round(float(2 * metrics.box.mp * metrics.box.mr / (metrics.box.mp + metrics.box.mr)), 4),
    "mAP50": round(float(metrics.box.map50), 4),
    "mAP50-95": round(float(metrics.box.map), 4),
}])

final_df = pd.concat([overall, df], ignore_index=True)
print(final_df.to_string(index=False))

final_df.to_csv("per_class_accuracy.csv", index=False)
print("\nSaved detailed report to per_class_accuracy.csv")


Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 100 layers, 9,422,475 parameters, 0 gradients, 21.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 276.8±175.0 MB/s, size: 7.4 KB)
val: Scanning /kaggle/working/Floor-Plan-AI-Object-Detection-1/valid/labels.cache... 1208 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1208/1208 337.8Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 8, len(boxes) = 1266. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 76/76 3.1it/s 24.7s0.3ss
                   all       1208       1266      0.982      0.889      0.917      0.865
Speed: 1.3ms preprocess, 15.8ms inference, 0.0ms loss, 0.9ms postprocess per image
Result